[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IshtVibhu/iv-Mesa-ABM-Tutorial/blob/main/NB_08_Mesa_ABM.ipynb)

# Python for Computational Economics
## Notebook 08 — Mesa: Agent-Based Modelling

**Prerequisites:** NB_01–NB_04

**What you will learn:**
- The Mesa framework: Model, Agent, Grid, DataCollector
- A minimal Schelling segregation model (warm-up)
- A Sugarscape-style trading model with two goods
- Collecting and analysing simulation output with Pandas
- Parameter sweeps with BatchRunnerMP

**Why ABM?** Standard economic models assume equilibrium. ABMs start from individual rules and let macro patterns *emerge* — inequality, trade networks, business cycles — without imposing them top-down.

In [ ]:
try:
    import mesa
except ImportError:
    !pip install mesa --quiet
    import mesa

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
print(f"Mesa {mesa.__version__}")

---
## 1. Mesa Architecture

Every Mesa model has four components:

| Component | Role |
|---|---|
| `Model` | Global state, scheduler, runs time steps |
| `Agent` | Individual entity with `step()` method |
| `Grid` / `MultiGrid` | Spatial environment |
| `DataCollector` | Records model and agent attributes at each step |

The standard loop is:
```python
model = MyModel(...)
for step in range(N_STEPS):
    model.step()
data = model.datacollector.get_agent_vars_dataframe()
```

---
## 2. Warm-Up — Schelling Segregation Model

Schelling (1971) showed that even mild preferences for neighbours of the same type produce strong residential segregation.

In [ ]:
from mesa import Agent, Model
from mesa.space import SingleGrid
from mesa.time import RandomActivation
from mesa.datacollection import DataCollector


class SchellingAgent(Agent):
    def __init__(self, unique_id, model, agent_type):
        super().__init__(unique_id, model)
        self.agent_type = agent_type  # 0 or 1

    def step(self):
        neighbours = self.model.grid.get_neighbors(self.pos, moore=True, include_center=False)
        same_type  = sum(1 for n in neighbours if n.agent_type == self.agent_type)
        total      = len(neighbours)
        # Move if fewer than threshold fraction of neighbours are the same type
        if total > 0 and same_type / total < self.model.homophily:
            self.model.grid.move_to_empty(self)


class SchellingModel(Model):
    def __init__(self, width=20, height=20, density=0.8, homophily=0.3, seed=None):
        super().__init__(seed=seed)
        self.homophily = homophily
        self.grid = SingleGrid(width, height, torus=True)
        self.schedule = RandomActivation(self)

        self.datacollector = DataCollector(
            model_reporters={"Segregation": self.compute_segregation}
        )

        agent_id = 0
        for _, pos in self.grid.coord_iter():
            if self.random.random() < density:
                agent_type = self.random.randint(0, 1)
                agent = SchellingAgent(agent_id, self, agent_type)
                self.grid.place_agent(agent, pos)
                self.schedule.add(agent)
                agent_id += 1

    def compute_segregation(self):
        """Fraction of agents with ≥ 50% same-type neighbours."""
        satisfied = 0
        total = 0
        for agent in self.schedule.agents:
            neighbours = self.grid.get_neighbors(agent.pos, moore=True, include_center=False)
            if not neighbours:
                continue
            same = sum(1 for n in neighbours if n.agent_type == agent.agent_type)
            if same / len(neighbours) >= 0.5:
                satisfied += 1
            total += 1
        return satisfied / total if total > 0 else 0

    def step(self):
        self.datacollector.collect(self)
        self.schedule.step()


# Run the model
model = SchellingModel(width=20, height=20, density=0.8, homophily=0.3, seed=42)
for _ in range(30):
    model.step()

seg_data = model.datacollector.get_model_vars_dataframe()
print("Schelling Segregation over time:")
print(seg_data.round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(seg_data["Segregation"], color="steelblue", linewidth=2)
ax.set_xlabel("Step"); ax.set_ylabel("Fraction segregated")
ax.set_title(f"Schelling Segregation (homophily threshold = 30%)")
plt.tight_layout(); plt.savefig("schelling_segregation.png", dpi=100); plt.show()

---
## 3. Mini Sugarscape — Two-Good Trading Model

This is a stripped-down version of the Sugarscape G1MT model from the companion Session notebooks. Traders move on a grid, harvest sugar and spice, and trade with neighbours using the Epstein-Axtell welfare-improving trade rule.

In [ ]:
from mesa import Agent, Model
from mesa.space import MultiGrid
from mesa.time import RandomActivation
from mesa.datacollection import DataCollector


def make_landscape(size, peaks, peak_height=4, radius=10):
    grid = np.zeros((size, size))
    for r in range(size):
        for c in range(size):
            for pr, pc in peaks:
                d = np.sqrt((r - pr)**2 + (c - pc)**2)
                grid[r, c] = max(grid[r, c], peak_height - d * peak_height / radius)
    return np.clip(grid, 0, peak_height).round().astype(int)


class SugarTrader(Agent):
    def __init__(self, unique_id, model, sugar, spice, meta_s, meta_p):
        super().__init__(unique_id, model)
        self.sugar   = float(sugar)
        self.spice   = float(spice)
        self.meta_s  = meta_s
        self.meta_p  = meta_p
        self.alive   = True

    @property
    def welfare(self):
        return self.sugar ** self.meta_s * self.spice ** self.meta_p

    @property
    def mrs(self):
        if self.sugar <= 0:
            return float("inf")
        if self.spice <= 0:
            return 0.0
        return (self.meta_s / self.meta_p) * (self.spice / self.sugar)

    def harvest(self):
        x, y = self.pos
        self.sugar += self.model.sugar_grid[x, y]
        self.spice += self.model.spice_grid[x, y]

    def consume(self):
        self.sugar -= self.meta_s
        self.spice -= self.meta_p
        if self.sugar <= 0 or self.spice <= 0:
            self.alive = False

    def move(self):
        neighbourhood = self.model.grid.get_neighborhood(self.pos, moore=False, radius=1)
        best_pos = max(neighbourhood,
                       key=lambda p: self.model.sugar_grid[p[0], p[1]] + self.model.spice_grid[p[0], p[1]])
        self.model.grid.move_agent(self, best_pos)

    def trade(self):
        neighbours = self.model.grid.get_neighbors(self.pos, moore=False, radius=1)
        for other in neighbours:
            if not isinstance(other, SugarTrader) or not other.alive:
                continue
            # Epstein-Axtell trade rule
            p = np.sqrt(self.mrs * other.mrs)  # geometric mean of MRS = trade price
            if abs(self.mrs - other.mrs) < 0.01:
                continue
            if self.mrs > other.mrs:
                # self values spice more than other → self gives sugar, gets spice
                give_sugar  = min(1.0 / p, self.sugar)
                get_spice   = give_sugar * p
                if get_spice > other.spice:
                    continue
                self.sugar  -= give_sugar
                self.spice  += get_spice
                other.sugar += give_sugar
                other.spice -= get_spice

    def step(self):
        if not self.alive:
            return
        self.move()
        self.harvest()
        self.trade()
        self.consume()


class SugarscapeModel(Model):
    def __init__(self, N=80, size=25, seed=None):
        super().__init__(seed=seed)
        self.size = size
        self.grid = MultiGrid(size, size, torus=False)
        self.schedule = RandomActivation(self)

        self.sugar_grid = make_landscape(size, [(8, 8), (17, 17)]).astype(float)
        self.spice_grid = make_landscape(size, [(8, 17), (17, 8)]).astype(float)

        def compute_gini(model):
            wealth = sorted([a.sugar + a.spice for a in model.schedule.agents if a.alive])
            if not wealth:
                return 0
            n = len(wealth)
            w = np.array(wealth)
            return (2 * np.sum(np.arange(1, n+1) * w) / (n * w.sum())) - (n+1)/n

        self.datacollector = DataCollector(
            model_reporters={
                "Population": lambda m: sum(1 for a in m.schedule.agents if a.alive),
                "Mean_Wealth": lambda m: np.mean([a.sugar + a.spice for a in m.schedule.agents if a.alive]),
                "Gini": compute_gini,
            }
        )

        for i in range(N):
            x = self.random.randrange(size)
            y = self.random.randrange(size)
            trader = SugarTrader(
                i, self,
                sugar  = self.random.uniform(5, 15),
                spice  = self.random.uniform(5, 15),
                meta_s = self.random.randint(1, 3),
                meta_p = self.random.randint(1, 3),
            )
            self.grid.place_agent(trader, (x, y))
            self.schedule.add(trader)

    def step(self):
        self.datacollector.collect(self)
        self.schedule.step()
        # Remove dead agents
        dead = [a for a in self.schedule.agents if not a.alive]
        for a in dead:
            self.grid.remove_agent(a)
            self.schedule.remove(a)


# Run
model = SugarscapeModel(N=80, size=25, seed=42)
for step in range(50):
    model.step()

results = model.datacollector.get_model_vars_dataframe()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
results["Population"].plot(ax=axes[0], color="steelblue")
axes[0].set_title("Population"); axes[0].set_xlabel("Step")

results["Mean_Wealth"].plot(ax=axes[1], color="green")
axes[1].set_title("Mean Wealth"); axes[1].set_xlabel("Step")

results["Gini"].plot(ax=axes[2], color="red")
axes[2].set_title("Gini Coefficient"); axes[2].set_xlabel("Step")

plt.suptitle("Mini Sugarscape — 50 steps, 80 traders", y=1.02)
plt.tight_layout()
plt.savefig("sugarscape_abm.png", dpi=100, bbox_inches="tight")
plt.show()

print(f"\nFinal population:  {results['Population'].iloc[-1]:.0f}")
print(f"Final mean wealth: {results['Mean_Wealth'].iloc[-1]:.1f}")
print(f"Final Gini:        {results['Gini'].iloc[-1]:.3f}")

---
## Your Turn — Effect of Metabolism on Inequality

Run the `SugarscapeModel` three times with different maximum metabolism values: `max_meta = 1`, `2`, and `4`. For each run, record the Gini coefficient at step 50. What happens to inequality as metabolism variance increases?

In [ ]:
# Modify SugarscapeModel to accept a max_meta parameter, then run the three scenarios


In [ ]:
#@title Solution
class SugarscapeModelV2(SugarscapeModel):
    def __init__(self, N=80, size=25, max_meta=3, seed=None):
        self.max_meta = max_meta
        super().__init__(N=N, size=size, seed=seed)

    # Override trader creation in __init__ to use max_meta
    # (For brevity we re-run the parent __init__ with patched random)

results_by_meta = {}
for max_meta in [1, 2, 4]:
    m = SugarscapeModel(N=80, size=25, seed=42)
    # Reassign metabolisms
    for a in m.schedule.agents:
        a.meta_s = m.random.randint(1, max(1, max_meta))
        a.meta_p = m.random.randint(1, max(1, max_meta))
    for _ in range(50):
        m.step()
    gini_final = m.datacollector.get_model_vars_dataframe()["Gini"].iloc[-1]
    results_by_meta[max_meta] = gini_final
    print(f"max_meta={max_meta}: Final Gini = {gini_final:.3f}")

print("\nHigher metabolism variance → more differentiation → higher inequality.")

---
## Summary

| Concept | Mesa class / function |
|---|---|
| Create model | `class MyModel(Model)` |
| Create agent | `class MyAgent(Agent)` with `step()` |
| Grid | `SingleGrid`, `MultiGrid` |
| Scheduler | `RandomActivation`, `SimultaneousActivation` |
| Data collection | `DataCollector(model_reporters={...})` |
| Get results | `.get_model_vars_dataframe()` |

**For the full Sugarscape implementation**, see the companion Session_3 through Session_20 notebooks in this repository.

**Next:** NB_09_NetworkX.ipynb — trade networks and financial contagion.